# 2. Price & Volatility — QA

**Purpose** — Validate price, returns, OHLCV, and realized vol before they feed IL, EVC, in-range probability, and backtests.

**Source tables**
- `mart_price_analytics_daily`
- `mart_price_intraday`
- `mart_lp_macro_daily`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


<function __main__.<lambda>()>

In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


## §1 — Data fetch & coverage

In [3]:
prices = db.read_query(f"""
SELECT pool, date, close_tick, close_price, has_swap, log_return,
       realized_vol_7d, realized_vol_30d, rolling_high, drawdown
FROM mart_price_analytics_daily
WHERE lower(pool) = lower('{POOL}')
  AND date <= toDate('{END_DATE}')
ORDER BY date
""")
intraday = db.read_query(f"""
SELECT pool, date, hour, open_tick, close_tick, open_price, high_price, low_price, close_price,
       volume_token0_equiv, swap_count, log_return_hourly, realized_vol_intraday
FROM mart_price_intraday
WHERE lower(pool) = lower('{POOL}')
  AND date >= toDate('{START_DATE_30D}')
  AND date <= toDate('{END_DATE}')
ORDER BY date, hour
""")
print(f"daily prices: {prices.height}  rows  ({prices['date'].min()} → {prices['date'].max()})")
print(f"intraday OHLCV: {intraday.height} rows ({intraday['date'].min()} → {intraday['date'].max()})")

daily prices: 1798  rows  (2021-05-05 → 2026-04-06)
intraday OHLCV: 744 rows (2026-03-07 → 2026-04-06)


## §2 — Sanity checks

In [4]:
# Date completeness on daily series
expected_days = (date.fromisoformat(END_DATE) - prices['date'].min()).days + 1
qa_check("daily price series contiguous (no missing dates)",
         prices.height == expected_days, severity="warn",
         detail=f"got={prices.height} expected={expected_days}")

n_zero_neg = (prices.filter(pl.col("close_price") <= 0)).height
qa_check("close_price > 0 on all rows", n_zero_neg == 0, detail=f"violations={n_zero_neg}")

n_neg_vol = (prices.filter((pl.col("realized_vol_7d") < 0) | (pl.col("realized_vol_30d") < 0))).height
qa_check("realized_vol_7d/30d non-negative", n_neg_vol == 0, detail=f"violations={n_neg_vol}")

🟢 PASS | daily price series contiguous (no missing dates)  —  got=1798 expected=1798
🟢 PASS | close_price > 0 on all rows  —  violations=0
🟢 PASS | realized_vol_7d/30d non-negative  —  violations=0


In [5]:
# Tail returns
tail = prices.filter(pl.col("log_return").abs() > 0.5)
qa_check("|log_return| > 50% extreme tail", tail.height == 0, severity="warn",
         detail=f"extreme_days={tail.height}")
if tail.height > 0:
    print(tail.select("date", "close_price", "log_return"))

# Vol ratio sanity
vr = prices.filter((pl.col("realized_vol_30d") > 0) & ((pl.col("realized_vol_7d") / pl.col("realized_vol_30d")) > 5))
qa_check("realized_vol_7d / realized_vol_30d <= 5",
         vr.height == 0, severity="warn", detail=f"days={vr.height}")

🟢 PASS | |log_return| > 50% extreme tail  —  extreme_days=0
🟢 PASS | realized_vol_7d / realized_vol_30d <= 5  —  days=0


In [6]:
# Intraday OHLC consistency
ohlc_q = f"""
SELECT count() AS n
FROM mart_price_intraday
WHERE lower(pool) = lower('{POOL}')
  AND date <= toDate('{END_DATE}')
  AND high_price < open_price OR high_price < close_price OR low_price > open_price OR low_price > close_price
"""
n_ohlc_bad = int(db.read_query(ohlc_q)["n"][0])
qa_check("intraday OHLC consistency (high>=o,c ; low<=o,c)", n_ohlc_bad == 0,
         detail=f"violations={n_ohlc_bad}")

# Hourly→daily reconciliation: end-of-day intraday close vs daily close
recon_q = f"""
WITH hourly_eod AS (
  SELECT pool, date, argMax(close_price, hour) AS eod_close
  FROM mart_price_intraday
  WHERE lower(pool) = lower('{POOL}') AND date >= toDate('{START_DATE_30D}') AND date <= toDate('{END_DATE}')
  GROUP BY pool, date
)
SELECT
  count() AS rows,
  countIf(abs(h.eod_close - p.close_price) / nullIf(p.close_price, 0) > 0.01) AS mismatches
FROM hourly_eod h
INNER JOIN mart_price_analytics_daily p USING (pool, date)
WHERE lower(p.pool) = lower('{POOL}')
"""
recon = db.read_query(recon_q).row(0, named=True)
qa_check("hourly EoD close ≈ daily close (within 1%)",
         int(recon["mismatches"]) == 0, severity="warn",
         detail=f"compared={recon['rows']} mismatches={recon['mismatches']}")

🟢 PASS | intraday OHLC consistency (high>=o,c ; low<=o,c)  —  violations=0
🟢 PASS | hourly EoD close ≈ daily close (within 1%)  —  compared=31 mismatches=0


## §3 — Diagnostic visuals

**QA purpose**: detect missing daily price observations. A gap = upstream swap pipeline missing data.

In [7]:
gaps = db.read_query(f"""
WITH bounds AS (
  SELECT min(date) AS min_d, max(date) AS max_d FROM mart_price_analytics_daily WHERE lower(pool) = lower('{POOL}')
), spine AS (
  SELECT addDays(min_d, toInt32(number)) AS d
  FROM bounds ARRAY JOIN range(toUInt32(dateDiff('day', min_d, toDate('{END_DATE}')) + 1)) AS number
)
SELECT s.d AS date,
       toUInt8(p.date IS NOT NULL) AS present
FROM spine s
LEFT JOIN mart_price_analytics_daily p ON p.date = s.d AND lower(p.pool) = lower('{POOL}')
WHERE s.d <= toDate('{END_DATE}')
ORDER BY s.d
""")
fig = go.Figure(go.Heatmap(
    z=[gaps["present"].to_list()],
    x=gaps["date"].to_list(),
    y=["price"],
    colorscale=[[0, "lightgray"], [1, "#22c55e"]],
    showscale=False,
))
fig.update_layout(height=120, title_text="Daily price coverage (1 = present)")
fig.show()
print(f"Missing days: {gaps.filter(pl.col('present') == 0).height}")

Missing days: 0


**QA purpose**: log-return distribution and tail audit. Tails > p99 should match known event days.

In [8]:
ret = prices.filter(pl.col("log_return").is_not_null())
fig = go.Figure(go.Histogram(x=ret["log_return"].to_list(), nbinsx=80))
p1 = ret["log_return"].quantile(0.01)
p99 = ret["log_return"].quantile(0.99)
fig.add_vline(x=p1,  line_dash="dash", line_color="red", annotation_text=f"p01={p1:.3f}")
fig.add_vline(x=p99, line_dash="dash", line_color="red", annotation_text=f"p99={p99:.3f}")
fig.update_layout(height=320, title_text="Daily log_return distribution")
fig.show()

**QA purpose**: realized vol term structure 7d vs 30d — compression / expansion regime audit.

In [9]:
recent = prices.filter(pl.col("date") >= date.fromisoformat(START_DATE_90D))
fig = go.Figure()
fig.add_trace(go.Scatter(x=recent["date"].to_list(), y=recent["realized_vol_7d"].to_list(),  name="σ 7d"))
fig.add_trace(go.Scatter(x=recent["date"].to_list(), y=recent["realized_vol_30d"].to_list(), name="σ 30d"))
fig.update_layout(height=320, title_text="Realized vol — 7d vs 30d (last 90d)")
fig.show()

**QA purpose**: validate intraday OHLCV last 90d. Candlestick lets you spot OHLC inversions visually.

In [10]:
recent = intraday.filter(pl.col("date") >= date.fromisoformat(START_DATE_30D))
fig = go.Figure(go.Candlestick(
    x=recent["hour"].to_list(),
    open=recent["open_price"].to_list(),
    high=recent["high_price"].to_list(),
    low=recent["low_price"].to_list(),
    close=recent["close_price"].to_list(),
))
fig.update_layout(height=400, title_text="Intraday OHLCV — last 30d (hourly)", xaxis_rangeslider_visible=False)
fig.show()


**QA purpose**: hour-of-day vol pattern — should reflect known activity hours; flat or spiked = data issue.

In [11]:
hod = intraday.with_columns(pl.col("hour").dt.hour().alias("hour_of_day")).group_by("hour_of_day").agg(
    pl.col("realized_vol_intraday").median().alias("median_intraday_vol"),
    pl.col("volume_token0_equiv").median().alias("median_vol_volume"),
).sort("hour_of_day")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Median intraday σ by hour", "Median volume by hour"))
fig.add_trace(go.Bar(x=hod["hour_of_day"].to_list(), y=hod["median_intraday_vol"].to_list()), row=1, col=1)
fig.add_trace(go.Bar(x=hod["hour_of_day"].to_list(), y=hod["median_vol_volume"].to_list()),    row=2, col=1)
fig.update_layout(height=420, showlegend=False, title_text="Intraday pattern (last 30d)")
fig.show()


## §4 — QA verdict

In [12]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: PASS
  PASS: 7   WARN: 0   FAIL: 0


label,status,severity,detail
str,str,str,str
"""daily price series contiguous …","""PASS""","""warn""","""got=1798 expected=1798"""
"""close_price > 0 on all rows""","""PASS""","""fail""","""violations=0"""
"""realized_vol_7d/30d non-negati…","""PASS""","""fail""","""violations=0"""
"""|log_return| > 50% extreme tai…","""PASS""","""warn""","""extreme_days=0"""
"""realized_vol_7d / realized_vol…","""PASS""","""warn""","""days=0"""
"""intraday OHLC consistency (hig…","""PASS""","""fail""","""violations=0"""
"""hourly EoD close ≈ daily close…","""PASS""","""warn""","""compared=31 mismatches=0"""
